# Importing necessary packages

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.ensemble import HistGradientBoostingClassifier
pd.set_option('display.max_columns', None)

# Loading data from csv files

In [2]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/raw/Train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Train.csv'
    },
    'test': {
        'local': '../data/raw/Test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Test.csv'
    },
    'var_defs': {
        'local': '../data/raw/VariableDefinitions.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/VariableDefinitions.csv'
    }
}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
var_defs = data['var_defs']


Loaded train from local path.
Loaded test from local path.
Loaded var_defs from local path.


## Checking for missing values

### Cleaning the training data

In [3]:
# Dropping missing rows with missing values
before = train.shape[0]
new_train = train.dropna(inplace=False)
after = new_train.shape[0]
print(f"Number of rows dropped :- {before - after}")

Number of rows dropped :- 1082


In [4]:
# Dropping duplicate rows with missing values
before = train.shape[0]
new_train = train.drop_duplicates()
after = new_train.shape[0]
print(f"Number of rows dropped :- {before - after}")

Number of rows dropped :- 0


### Cleaning the testing data

In [5]:
# Dropping missing rows with missing values
before = test.shape[0]
new_test = test.dropna(inplace=False)
after = new_test.shape[0]
print(f"Number of rows dropped :- {before - after}")

Number of rows dropped :- 364


In [6]:
# Dropping duplicate rows with missing values
before = test.shape[0]
new_test = test.drop_duplicates()
after = new_test.shape[0]
print(f"Number of rows dropped :- {before - after}")

Number of rows dropped :- 0


## Setting up columns

In [7]:
target_col = 'cost_category'
id_col = 'Tour_ID'
target_classes = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']

# Feature Engineering

In [8]:
def preprocess_data(df):
    df = df.copy()
    # Fix data entry typos
    df['main_activity'] = df['main_activity'].replace({'Widlife Tourism': 'Wildlife Tourism'})

    df['total_people'] = df['total_female'] + df['total_male']
    df['total_nights'] = df['night_mainland'] + df['night_zanzibar']
    
    # Package inclusions sum
    package_cols = [
        'package_transport_int', 'package_accomodation', 'package_food',
        'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
        'package_insurance'
    ]
    df['package_count'] = (df[package_cols] == 'Yes').sum(axis=1)
    
    return df

In [9]:
# Applying processing
train_df = preprocess_data(train)
test_df = preprocess_data(test)

## Encoding the categorical feature columns
- Categorical Column Preparation

In [10]:
cat_cols = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity', 
    'info_source', 'tour_arrangement', 'package_transport_int', 
    'package_accomodation', 'package_food', 'package_transport_tz', 
    'package_sightseeing', 'package_guided_tour', 'package_insurance', 'first_trip_tz'
]

X = train_df.drop(columns=[id_col, target_col])
y = train_df[target_col]
X_test = test_df.drop(columns=[id_col])

# Convert text/object columns to pandas 'category' type for gradient boosting
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [ ]:
# ---------------------------------------------------------
# Step 4: Stratified Cross-Validation & Modeling
# ---------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros((len(train_df), len(target_classes)))
test_preds = np.zeros((len(test_df), len(target_classes)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # Classifier native to categorical feature handling
    model = HistGradientBoostingClassifier(
        categorical_features=cat_cols,
        max_iter=300,
        learning_rate=0.05,
        random_state=42
    )
    
    model.fit(X_tr, y_tr)
    
    # Out-of-fold validation prediction
    oof_preds[val_idx] = model.predict_proba(X_va)
    # Average test set predictions across folds
    test_preds += model.predict_proba(X_test) / skf.n_splits

# Print Out-of-Fold Validation Metric
print(f"OOF Multi-Class Log Loss: {log_loss(y, oof_preds):.4f}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Format & Export Submission File
# ---------------------------------------------------------
# Map probability array back to class columns matching sample submission
submission = pd.DataFrame(test_preds, columns=model.classes_)
submission.insert(0, id_col, test_df[id_col])

# Strict column ordering as expected by Zindi
target_order = ['Tour_ID', 'High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']
submission = submission[target_order]

In [ ]:
submission